In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
df = pd.read_csv("IMDb Movies India.csv", encoding="latin1")

df.head()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [4]:
print("Shape:", df.shape)

df.info()

print("\nMissing Values:\n")
print(df.isnull().sum())

Shape: (15509, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB

Missing Values:

Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64


In [5]:
df = df.dropna(subset=["Rating"])

df["Genre"] = df["Genre"].fillna("")
df["Director"] = df["Director"].fillna("")
df["Actor 1"] = df["Actor 1"].fillna("")
df["Actor 2"] = df["Actor 2"].fillna("")
df["Actor 3"] = df["Actor 3"].fillna("")

In [6]:
df["Cast"] = (
    df["Actor 1"] + " " +
    df["Actor 2"] + " " +
    df["Actor 3"]
)

df[["Genre", "Director", "Cast", "Rating"]].head()

,Genre,Director,Cast,Rating
1,Drama,Gaurav Bakshi,Rasika Dugal Vivek Ghamande Arvind Jangid,7.0
3,"Comedy, Romance",Ovais Khan,Prateik Ishita Raj Siddhant Kapoor,4.4
5,"Comedy, Drama, Musical",Rahul Rawail,Bobby Deol Aishwarya Rai Bachchan Shammi Kapoor,4.7
6,"Drama, Romance, War",Shoojit Sircar,Jimmy Sheirgill Minissha Lamba Yashpal Sharma,7.4
8,"Horror, Mystery, Thriller",Allyson Patel,Yash Dave Muntazir Ahmad Kiran Bhatia,5.6


In [7]:
X = df[["Genre", "Director", "Cast"]]

y = df["Rating"]

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("genre", TfidfVectorizer(), "Genre"),
        ("director", TfidfVectorizer(), "Director"),
        ("cast", TfidfVectorizer(), "Cast")
    ]
)

In [9]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=50,
        random_state=42
    ))
])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [12]:
predictions = model.predict(X_test)

In [13]:
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", round(mae, 3))
print("R2 Score:", round(r2, 3))

Mean Absolute Error: 0.926
R2 Score: 0.23


In [14]:
sample_movie = pd.DataFrame({
    "Genre": ["Drama"],
    "Director": ["Rajkumar Hirani"],
    "Cast": ["Aamir Khan Kareena Kapoor"]
})

predicted_rating = model.predict(sample_movie)

print("Predicted Rating:", round(predicted_rating[0], 2))

Predicted Rating: 6.8


In [15]:
results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": predictions
})

results.head(10)

,Actual Rating,Predicted Rating
0,3.3,5.142000
1,5.3,6.220000
2,5.7,5.480000
3,7.2,6.628000
4,3.5,4.538000
5,7.2,5.854000
6,3.8,6.518000
7,6.9,6.025333
8,5.2,6.272000
9,7.4,7.132000
